# Projet 3 — Stratégie *k-nearest neighbors* pour l'allocation de portefeuille

**M2 ISF — Gestion de risques et construction de portefeuille (G. Turinici), 2026/27**

**Auteurs :** Albert LI, Antonin DUROUSSEAU, Ikram MERZOUG, Jathiss JEGATHEESWARAN

| Membre | Contribution exacte |
|---|---|
| Albert LI | 0.25 |
| Antonin DUROUSSEAU | 0.25 |
| Ikram MERZOUG | 0.25 |
| Jathiss JEGATHEESWARAN | 0.25 |


In [ ]:
# ===================== Imports et configuration =====================
import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from sklearn.covariance import LedoitWolf


# 1. Rappel : la méthode des *k* plus proches voisins

Le k-NN est une méthode d'apprentissage supervisé **non paramétrique** pour prédire la sortie associée à un point $x$, on cherche dans la base d'exemples les $k$ points les plus proches de $x$ selon une distance $d$, puis on agrège leurs sorties.

**Application à l'allocation**
- un exemple = une date passée $s$ ;
- l'entrée = l'**état du marché** $X_s$ (vecteur de caractéristiques) ;
- la sortie = l'allocation $w^*_s$ qui *aurait été* optimale (Markowitz) sur la fenêtre future $[s+1, s+H]$.

À la date $t$, on cherche les $k$ dates passées $s_1,\dots,s_k$ dont l'état ressemble le plus à $X_t$, et on investit dans la moyenne pondérée de leurs allocations optimales :

Chaque allocation $w^*_s$ est positif et somme à 1 ; une moyenne pondérée de tels vecteurs l'est aussi : $\hat w_t$ est donc automatiquement un portefeuille admissible (long-only, investi à 100 %).

## 1. Les deux marchés choisis ainsi que la période d'étude

**Marché 1 — CAC 40 (d = 10 actions)** : LVMH, L'Oréal, Sanofi, BNP Paribas, TotalEnergies, Schneider Electric, Air Liquide, Vinci, AXA, STMicroelectronics.
- 10 secteurs différents (luxe, consommation, santé, banque, énergie, industrie, chimie, BTP, assurance, semi-conducteurs) : il faut des actifs qui réagissent différemment selon le régime, sinon il n'y a rien à allouer.

**Marché 2 — Chine : actions cotées à Hong Kong (d = 10)**, en dollars de Hong Kong (indexé sur l'USD) :

| Secteur | Société | Ticker |
|---|---|---|
| Internet | Tencent | 0700.HK |
| Télécoms | China Mobile | 0941.HK |
| Énergie | CNOOC | 0883.HK |
| Assurance | Ping An | 2318.HK |
| Banque | ICBC | 1398.HK |
| Automobile / batteries | BYD | 1211.HK |
| Consommation discrétionnaire | Anta Sports | 2020.HK |
| Santé | CSPC Pharmaceutical | 1093.HK |
| Immobilier | China Overseas Land | 0688.HK |
| Consommation courante | China Resources Beer | 0291.HK |

- **Même structure que le CAC 40** (10 actions de 10 secteurs différents) : les résultats des deux marchés sont comparables, et un éventuel écart ne vient pas d'une différence de type d'actif.

Les deux marchés sont traités **séparément**.

**Période d'étude : 2008 – 2023**, précédée d'un historique long

| Période | Régime de marché |
|---|---|
| 2005 (CAC) / 2007 (Chine) – 2009 | constitution de l'historique, crise financière de 2008 |
| 2010 – 2017 | crise de l'euro (2011), krach chinois (2015-16), hausse régulière |
| 2018 – 2019 | croissance puis guerre commerciale sino-américaine |
| 2020 | Covid : krach brutal et volatilité extrême |
| 2021 – 2022 | reprise, choc inflationniste et hausse des taux (rotation value/growth), régulation de la tech chinoise |
| 2023 | normalisation, forte dispersion entre titres |

- La période 2018-2023 traverse quatre régimes très différents en six ans : c'est un banc d'essai exigeant pour une méthode censée s'adapter au régime.
- **Pourquoi un historique depuis 2005/2007** : un k-NN ne peut proposer comme voisin qu'une situation qu'il a déjà vue. Pour reconnaître le krach de 2020, il faut que la base contienne celui de 2008.

**Sources et reproductibilité**
- Prix : Yahoo Finance via `yfinance` (cours ajustés des dividendes et splits). Toutes les variables, y compris les facteurs de style, sont calculées à partir de ces prix : aucune autre source n'est nécessaire.
- Au premier lancement, les prix sont téléchargés puis sauvegardés en CSV dans `data/`. Ensuite, le notebook lit uniquement ces CSV (fournis dans le zip) : il s'exécute sans connexion ni intervention.

In [ ]:
SEED = 2026                                   # graine fixée : résultats reproductibles
np.random.seed(SEED)
DATA_DIR, RES_DIR = "data", "results"         # données brutes / résultats (cibles, tableaux, figures)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
pd.set_option("display.float_format", "{:.4f}".format)

END = "2025-12-31"     # fin de la période d'étude (2008-2023)
REB = 5                # rééquilibrage toutes les 5 séances (hebdomadaire)
MIN_GAP = 10           # écart minimal (en séances) entre deux voisins retenus
GAMMA = 5              # aversion au risque de la cible moyenne-variance
K_GRID = [5, 10, 20, 40]
METRICS = ["euclidienne", "manhattan", "mahalanobis"]
H_GRID = [20, 60]      # horizon de la fenêtre future (≈ 1 et 3 mois)

# value / growth : paniers servant à construire le proxy du facteur HML (§2)
END = "2025-12-31"

MARKETS = {
    "CAC40": dict(
        tickers=[
            "MC.PA", "OR.PA", "SAN.PA", "BNP.PA", "TTE.PA",
            "SU.PA", "AI.PA", "DG.PA", "CS.PA", "STMPA.PA"
        ],
        value=["BNP.PA", "TTE.PA", "CS.PA", "DG.PA"],
        growth=["MC.PA", "OR.PA", "SU.PA", "STMPA.PA"],
        start="2008-01-01",
        val=("2008-01-01", "2017-12-31"),   # Période de calibration / recherche de k
        test=("2018-01-01", END)            # Période de test Out-of-Sample
    ),
    "Chine": dict(
        tickers=[
            "0700.HK", "0941.HK", "0883.HK", "2318.HK", "1398.HK",
            "1211.HK", "2020.HK", "1093.HK", "0688.HK", "0291.HK"
        ],
        value=["1398.HK", "0883.HK", "0941.HK", "0688.HK"],
        growth=["0700.HK", "1211.HK", "2020.HK", "1093.HK"],
        start="2008-01-01",
        val=("2008-01-01", "2017-12-31"),
        test=("2018-01-01", END)
    ),
}

In [ ]:
#Pour supprimer les caches
import os
for name in ["CAC40", "Chine"]:
    p = f"{DATA_DIR}/prices_{name}.csv"
    if os.path.exists(p):
        os.remove(p)

In [ ]:
# ===================== Chargement des prix (avec cache CSV) =====================
def load_prices(name, tickers, start):
    """Prix de clôture ajustés. Téléchargés une seule fois, puis relus depuis data/."""
    path = f"{DATA_DIR}/prices_{name}.csv"
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=True)
    import yfinance as yf                                  # nécessaire uniquement au 1er lancement
    end_dl = (pd.Timestamp(END) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")   # borne exclue par yfinance
    px = yf.download(tickers, start=start, end=end_dl, auto_adjust=True, progress=False)["Close"]
    px = px[tickers].dropna()                              # dates où tous les actifs cotent
    px.to_csv(path)
    return px


for name, M in MARKETS.items():
    M["px"] = load_prices(name, M["tickers"], M["start"])
    print(f"{name:6s}: {M['px'].shape[1]} actifs, prix du {M['px'].index[0].date()} au {M['px'].index[-1].date()}")

CAC40 : 10 actifs, prix du 2008-01-02 au 2025-12-31
Chine : 10 actifs, prix du 2008-01-02 au 2025-12-31


## 2. Description des caractéristiques du Dataset géneré

L'état du marché est modélisé à travers un espace de caractéristiques regroupant différentes catégories de données. L'association d'indicateurs spécifiques par actions et de facteurs macro-financiers globaux permettent de construire le vecteur d'état à chaque date d'observation.

### 2.1 Variables spécifiques par action

Afin de capturer la dynamique individuelle de chaque titre, trois indicateurs financiers ont été calculés. Ils génèrent 30 colonnes du vecteur d'état et permettent à l'algorithme d'identifier des régimes de marché similaires.

$$
\begin{array}{|l|c|c|l|}
\hline
\textbf{Indicateur} & \textbf{Formule} & \textbf{Fenêtre temporelle} & \textbf{Il quantifie} \\
\hline
\texttt{Ret20} & \dfrac{P_t}{P_{t-20}} - 1 & 20 \text{ jours} & \text{la performance du titre} \\[10pt]
\hline
\texttt{Vol60} & \sigma_{60} \times \sqrt{252} & 60 \text{ jours} & \text{le risque de l'action} \\[6pt]
\hline
\texttt{EMA10\_50} & \dfrac{\text{EMA}_{10}(P)_t}{\text{EMA}_{50}(P)_t} & 10 \text{ et } 50 \text{ jours} & \text{la direction de la tendance} \\[10pt]
\hline
\end{array}
$$

Ces indicateurs reposent sur les fomules suivantes :

* **La volatilité ($\sigma_N$) :** Elle correspond à l'écart-type empirique des rendements quotidiens $R_i$ sur la fenêtre étudiée (avec $\bar{R}$ le rendement moyen sur la période).
  $$ \sigma_N = \sqrt{\frac{1}{N-1} \sum_{i=1}^N (R_i - \bar{R})^2} $$

* **La Moyenne Mobile Exponentielle ($\text{EMA}_N$) :** Elle se calcule de manière récursive. Contrairement à une moyenne arithmétique classique, elle applique un coefficient de lissage $\alpha = \frac{2}{N+1}$ qui donne davantage de poids aux prix les plus récents ($P_t$).
  $$ \text{EMA}_t = \alpha P_t + (1 - \alpha) \text{EMA}_{t-1} $$

### 2.2 Les facteurs macro-financiers globaux

Pour compléter l'analyse individuelle des titres, 4 facteurs de marché transversaux sont ajoutés. Ils permettent à l'algorithme de comprendre le contexte économique global à la date étudiée.

| Facteur | Description | Utilité dans le modèle |
| :--- | :--- | :--- |
| **Market Momentum** | Rendement de l'indice de référence sur 20 jours | Indique si le marché global est en phase de hausse ou de baisse. |
| **Volatilité du Marché** | Volatilité réalisée de l'indice sur 60 jours | Mesure le niveau d'incertitude ou de stress général. |
| **Facteur de Style** | Différence de performance entre les actions "Value" et "Growth" | Capte les rotations sectorielles et les variations de l’aversion au risque des investisseurs. |
| **Dispersion Sectorielle** | Écart-type des rendements des 10 actions | Évalue si les actions évoluent toutes dans le même sens ou de manière désynchronisée. |


### 2.3 Exemple de vecteur d'état à une date $t$

À une date d'observation donnée (par exemple le 15 novembre 2023), l'environnement de marché est figé sous la forme d'un unique vecteur ligne $X_t \in \mathbb{R}^{34}$.

Ce vecteur concatène les 3 indicateurs pour chacune des 10 actions (ex: Air Liquide, LVMH, Sanofi, etc.), suivis des 4 facteurs macro-financiers globaux :

$$
X_t = \begin{bmatrix}
\text{Ret20}_{\text{Actif 1}} \\
\text{Vol60}_{\text{Actif 1}} \\
\text{EMA10_50}_{\text{Actif 1}} \\
\vdots \\
\text{Ret20}_{\text{Actif 10}} \\
\text{Vol60}_{\text{Actif 10}} \\
\text{EMA10_50}_{\text{Actif 10}} \\
\text{Market Momentum} \\
\text{Volatilité du Marché} \\
\text{Facteur HML} \\
\text{Dispersion Sectorielle}
\end{bmatrix}^T
$$

In [ ]:
# ===================== Construction du vecteur d'état =====================
def build_features(px, value, growth):
    """X_t : une ligne par date, colonnes nommées 'famille|variable' (valeurs brutes, non standardisées)."""
    r = px.pct_change()
    ret20 = px.pct_change(20)                                             # rendement 20 séances
    vol60 = r.rolling(60).std() * np.sqrt(252)                            # volatilité 60 séances annualisée
    ema = px.ewm(span=10, adjust=False).mean() / px.ewm(span=50, adjust=False).mean()   # ratio EMA10/EMA50
    ff = pd.DataFrame({                                                   # facteurs de style (proxys)
        "FF|MKT_Momentum": ret20.mean(axis=1),
        "FF|HML_ValueVsGrowth": ret20[value].mean(axis=1) - ret20[growth].mean(axis=1),
        "FF|MKT_Volatility": vol60.mean(axis=1),
        "FF|MKT_Dispersion": ret20.std(axis=1)})
    X = pd.concat([ret20.add_prefix("Ret20|"), vol60.add_prefix("Vol60|"),
                   ema.add_prefix("EMA10_50|"), ff], axis=1)
    return X.dropna()                                                     # retire la période de chauffe


def family_corr(X):
    """|corrélation| moyenne entre familles de variables (diagnostic de redondance)."""
    fam = X.columns.str.split("|").str[0]
    C = X.corr().abs().values.copy()
    np.fill_diagonal(C, np.nan)
    C = pd.DataFrame(C, index=fam, columns=fam)
    return C.groupby(level=0).mean().T.groupby(level=0).mean()


for name, M in MARKETS.items():
    M["X"] = build_features(M["px"], M["value"], M["growth"])
    M["R"] = M["px"].pct_change().loc[M["X"].index]          # rendements alignés sur X
    M["labels"] = {}                                         # cache des allocations cibles
    fam = M["X"].columns.str.split("|").str[0]
    print(f"{name:6s}: X de dimension {M['X'].shape} | {M['X'].index[0].date()} -> {M['X'].index[-1].date()}"
          f" | variables par famille : { {f: int(n) for f, n in fam.value_counts().sort_index().items()} }")
    print(f"\n|corrélation| moyenne entre familles — {name}")
    display(family_corr(M["X"]).round(2))

CAC40 : X de dimension (4546, 34) | 2008-03-28 -> 2025-12-31 | variables par famille : {'EMA10_50': 10, 'FF': 4, 'Ret20': 10, 'Vol60': 10}

|corrélation| moyenne entre familles — CAC40


,EMA10_50,FF,Ret20,Vol60
EMA10_50,0.4900,0.3000,0.4100,0.2400
FF,0.3000,0.1400,0.2700,0.3600
Ret20,0.4100,0.2700,0.5000,0.0400
Vol60,0.2400,0.3600,0.0400,0.7600


Chine : X de dimension (4378, 34) | 2008-04-01 -> 2025-12-31 | variables par famille : {'EMA10_50': 10, 'FF': 4, 'Ret20': 10, 'Vol60': 10}

|corrélation| moyenne entre familles — Chine


,EMA10_50,FF,Ret20,Vol60
EMA10_50,0.3900,0.2500,0.3400,0.1300
FF,0.2500,0.2400,0.2800,0.3200
Ret20,0.3400,0.2800,0.3800,0.0400
Vol60,0.1300,0.3200,0.0400,0.6600


### Aperçu du dataset

Les valeurs affichées sont brutes (avant standardisation) : rendements et volatilités en décimal, ratios EMA autour de 1. Le dataset de chaque marché est aussi exporté dans `results/` pour être consulté dans Excel.

In [ ]:
# ===================== Aperçu et export du dataset =====================
pd.set_option("display.max_columns", 40)
for name, M in MARKETS.items():
    print(f"\n===== {name} : premières lignes de X =====")
    display(M["X"].head())
    display(M["X"].describe().T[["mean", "std", "min", "50%", "max"]])
    path = f"{RES_DIR}/dataset_knn_{name}.xlsx"
    try:
        M["X"].to_excel(path)                                 # nécessite openpyxl
    except ImportError:
        path = path.replace(".xlsx", ".csv")
        M["X"].to_csv(path)
    print("Dataset exporté :", path)


===== CAC40 : premières lignes de X =====


,Ret20|MC.PA,Ret20|OR.PA,Ret20|SAN.PA,Ret20|BNP.PA,Ret20|TTE.PA,Ret20|SU.PA,Ret20|AI.PA,Ret20|DG.PA,Ret20|CS.PA,Ret20|STMPA.PA,Vol60|MC.PA,Vol60|OR.PA,Vol60|SAN.PA,Vol60|BNP.PA,Vol60|TTE.PA,Vol60|SU.PA,Vol60|AI.PA,Vol60|DG.PA,Vol60|CS.PA,Vol60|STMPA.PA,EMA10_50|MC.PA,EMA10_50|OR.PA,EMA10_50|SAN.PA,EMA10_50|BNP.PA,EMA10_50|TTE.PA,EMA10_50|SU.PA,EMA10_50|AI.PA,EMA10_50|DG.PA,EMA10_50|CS.PA,EMA10_50|STMPA.PA,FF|MKT_Momentum,FF|HML_ValueVsGrowth,FF|MKT_Volatility,FF|MKT_Dispersion
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2008-03-28,-0.0318,-0.0194,-0.0861,0.0089,-0.0795,0.0096,-0.0184,-0.0313,-0.0252,-0.2264,0.3950,0.4292,0.3126,0.5006,0.3246,0.4902,0.3721,0.4682,0.5229,0.4179,0.9729,0.9761,0.9153,0.9867,0.9516,1.0179,0.9973,0.9905,0.9744,0.8768,-0.0499,0.0352,0.4233,0.0696
2008-03-31,0.0090,0.0108,-0.0575,0.0412,-0.0670,0.0539,0.0098,-0.0091,-0.0043,-0.1860,0.3965,0.4287,0.3126,0.5008,0.3201,0.4896,0.3731,0.4604,0.5232,0.4197,0.9792,0.9777,0.9202,0.9919,0.9528,1.0189,1.0007,0.9942,0.9816,0.8808,-0.0199,0.0183,0.4225,0.0695
2008-04-01,0.0733,0.0476,-0.0039,0.1373,-0.0456,0.1005,0.0459,0.0246,0.0912,-0.1112,0.4009,0.4323,0.3174,0.5161,0.3219,0.4820,0.3750,0.4614,0.5406,0.4299,0.9899,0.9826,0.9276,1.0048,0.9561,1.0219,1.0059,1.0012,0.9971,0.8919,0.0360,0.0243,0.4277,0.0739
2008-04-02,0.0843,0.0616,0.0110,0.1907,-0.0222,0.1276,0.0617,0.0372,0.1527,-0.0387,0.4010,0.4323,0.3130,0.5202,0.3207,0.4752,0.3752,0.4442,0.5454,0.4429,0.9990,0.9860,0.9364,1.0196,0.9613,1.0249,1.0109,1.0085,1.0149,0.9081,0.0666,0.0309,0.4270,0.0744
2008-04-03,0.0643,0.0530,0.0184,0.1738,0.0018,0.0829,0.0752,0.0713,0.1472,-0.0350,0.4042,0.4312,0.2989,0.5214,0.3208,0.4778,0.3751,0.4334,0.5499,0.4494,1.0024,0.9869,0.9428,1.0281,0.9671,1.0233,1.0154,1.0150,1.0232,0.9157,0.0653,0.0573,0.4262,0.0627


,mean,std,min,50%,max
Ret20|MC.PA,0.0142,0.0742,-0.3129,0.0146,0.2630
Ret20|OR.PA,0.0098,0.0558,-0.2239,0.0093,0.1970
Ret20|SAN.PA,0.0074,0.0575,-0.2499,0.0088,0.2458
Ret20|BNP.PA,0.0100,0.0996,-0.5088,0.0118,0.6462
Ret20|TTE.PA,0.0075,0.0681,-0.5162,0.0071,0.4949
Ret20|SU.PA,0.0129,0.0739,-0.3462,0.0141,0.3020
Ret20|AI.PA,0.0090,0.0515,-0.2903,0.0082,0.2145
Ret20|DG.PA,0.0097,0.0666,-0.4660,0.0117,0.3414
Ret20|CS.PA,0.0127,0.0960,-0.5706,0.0153,0.7368
Ret20|STMPA.PA,0.0143,0.1151,-0.5197,0.0122,0.5324


Dataset exporté : results/dataset_knn_CAC40.xlsx

===== Chine : premières lignes de X =====


,Ret20|0700.HK,Ret20|0941.HK,Ret20|0883.HK,Ret20|2318.HK,Ret20|1398.HK,Ret20|1211.HK,Ret20|2020.HK,Ret20|1093.HK,Ret20|0688.HK,Ret20|0291.HK,Vol60|0700.HK,Vol60|0941.HK,Vol60|0883.HK,Vol60|2318.HK,Vol60|1398.HK,Vol60|1211.HK,Vol60|2020.HK,Vol60|1093.HK,Vol60|0688.HK,Vol60|0291.HK,EMA10_50|0700.HK,EMA10_50|0941.HK,EMA10_50|0883.HK,EMA10_50|2318.HK,EMA10_50|1398.HK,EMA10_50|1211.HK,EMA10_50|2020.HK,EMA10_50|1093.HK,EMA10_50|0688.HK,EMA10_50|0291.HK,FF|MKT_Momentum,FF|HML_ValueVsGrowth,FF|MKT_Volatility,FF|MKT_Dispersion
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2008-04-01,-0.0986,-0.0192,-0.1456,-0.0840,0.0018,-0.0718,-0.0490,0.0167,-0.0718,-0.1115,0.6247,0.5229,0.7856,0.7439,0.6596,0.7764,0.5815,0.7533,0.8226,0.6998,0.9298,0.9733,0.9478,0.9120,1.0196,1.0426,0.9042,0.9685,0.9720,0.9105,-0.0633,-0.0080,0.6970,0.0513
2008-04-02,-0.0622,0.0474,-0.0679,-0.0042,0.0921,0.0056,0.0059,0.0041,-0.0081,-0.0389,0.6268,0.5254,0.7875,0.7519,0.6541,0.7747,0.5878,0.7496,0.8208,0.7010,0.9375,0.9842,0.9541,0.9244,1.0328,1.0473,0.9200,0.9797,0.9850,0.9234,-0.0026,0.0275,0.6980,0.0481
2008-04-03,-0.0103,0.0888,0.0083,0.1259,0.1310,-0.0098,-0.0143,-0.0198,0.0887,-0.0283,0.6207,0.5252,0.7822,0.7693,0.6536,0.7752,0.5735,0.7409,0.8267,0.6931,0.9501,0.9955,0.9644,0.9454,1.0443,1.0490,0.9293,0.9904,1.0026,0.9330,0.0360,0.0927,0.6960,0.0645
2008-04-07,0.0114,0.1224,0.0437,0.1723,0.1346,0.0340,0.0180,-0.0353,0.1335,0.0309,0.6187,0.5307,0.7813,0.7763,0.6532,0.7703,0.5647,0.7400,0.8257,0.6970,0.9622,1.0100,0.9751,0.9702,1.0538,1.0485,0.9404,0.9977,1.0179,0.9462,0.0665,0.1015,0.6958,0.0683
2008-04-08,0.0063,0.0923,0.0065,0.1076,0.1205,0.0693,0.0204,-0.0417,0.0811,-0.0133,0.6181,0.5328,0.7802,0.7766,0.6534,0.7687,0.5649,0.7421,0.8277,0.6988,0.9703,1.0176,0.9828,0.9873,1.0597,1.0531,0.9502,1.0076,1.0255,0.9529,0.0449,0.0615,0.6963,0.0560


,mean,std,min,50%,max
Ret20|0700.HK,0.0248,0.0989,-0.3660,0.0218,0.5067
Ret20|0941.HK,0.0046,0.0600,-0.3304,0.0031,0.2870
Ret20|0883.HK,0.0121,0.0963,-0.5629,0.0100,0.4605
Ret20|2318.HK,0.0114,0.1069,-0.5437,0.0054,0.7009
Ret20|1398.HK,0.0081,0.0737,-0.4017,0.0035,0.4114
Ret20|1211.HK,0.0271,0.1685,-0.4476,0.0010,1.1320
Ret20|2020.HK,0.0208,0.1216,-0.5165,0.0188,0.6298
Ret20|1093.HK,0.0185,0.1272,-0.5000,0.0115,0.6632
Ret20|0688.HK,0.0072,0.0985,-0.3927,0.0003,0.5587
Ret20|0291.HK,0.0104,0.1070,-0.4552,0.0023,0.6422


Dataset exporté : results/dataset_knn_Chine.xlsx


## 3. Mise à l'échelle et choix de la distance

**Mise à l'échelle** : C'est indispensable, car on a des rendements calculés sur une fenetre de 20j, une volatilité décimale et un ratio qui n'ont ni la même unité ni la même amplitude. Sans elle, la distance serait dominée par les variables aux plus grandes valeurs.
1. Nous effectuons donc un **z-score** avec moyenne et écart-type calculés uniquement sur l'historique disponible
2. **Pondération par famille** : chaque colonne est divisée par $\sqrt{n_{\text{famille}}}$, de sorte que chacune des 4 familles (Ret20, Vol60, EMA10_50, FF) pèse autant dans la distance. Sans cela, les 30 colonnes par action écraseraient les 4 facteurs de style.

| Distance | &emsp;&emsp;&emsp;&emsp; Formule &emsp;&emsp;&emsp;&emsp; | Avantages | Inconvénients |
| :--- | :---: | :--- | :--- |
| **Euclidienne** | $\sqrt{\sum_{j} (x_j - y_j)^2}$ | Référence simple ; pénalise fortement les gros écarts | Sensible aux valeurs extrêmes et aux variables redondantes |
| **Manhattan** | $\sum_{j} \vert x_j - y_j \vert$ | Plus robuste aux écarts extrêmes | Ne tient pas compte des corrélations |
| **Mahalanobis** | $\sqrt{(x - y)^T \Sigma^{-1} (x - y)}$ | Tient compte des corrélations et des échelles | Inversion de la matrice de covariance requise |


*Remarques:*

Une distance euclidienne compte donc plusieurs fois la même information tandis que Mahalanobis la corrige

- La distance de Mahalanobis est invariante à tout changement d'échelle linéaire des variables : elle se standardise d'elle-même, et la pondération par famille n'a pas d'effet sur elle.

**Écart minimal entre voisins** : les variables sont des moyennes glissantes, donc deux jours consécutifs ont presque le même $X$. On impose donc un écart d'au moins 10 séances entre deux voisins retenus.

## 4. Calcul des allocations des k voisins

Pour entraîner notre algorithme, nous devons associer à chaque date historique $s$ une "cible" : l'allocation qui se serait révélée optimale sur la fenêtre d'investissement future $[s+1, s+H]$.

Nous avons opté pour une approche de Markowitz classique (Moyenne-Variance) sous contrainte *long-only* (poids strictement positifs ou nuls, dont la somme vaut 1).

Pour un portefeuille défini par le vecteur de pondérations $w$, le rendement espéré $\mu_p$ et la variance (risque) $\sigma_p^2$ s'écrivent :
$$ \mu_p = \mu_s^\top w \quad \text{et} \quad \sigma_p^2 = w^\top\Sigma_s w $$

où $\mu_s$ et $\Sigma_s$ sont respectivement le vecteur des rendements moyens et la matrice de covariance empirique annualisés, calculés directement sur la fenêtre future.

Notre algorithme recherche l'allocation $w^*_s$ qui maximise la fonction d'utilité avec un coefficient d'aversion au risque modéré ($\gamma=5$) :
$$ w^*_s = \arg\max_{w\ge 0,\ \mathbf 1^\top w=1}\ \left( \mu_p - \frac{\gamma}{2}\sigma_p^2 \right) $$


**`optimize_markowitz(returns_window, gamma)`** calcule l'allocation optimale sur une fenêtre de rendements :
1. La fonction estime le rendement moyen $\mu$ et la covariance $\Sigma$, **annualisés** ($\times 252$) ;
2. définit l'objectif à **minimiser**, l'opposé de l'utilité : $\frac{\gamma}{2}w^\top\Sigma w-\mu^\top w$;
3. fournit la **jacobienne** (le gradient $\gamma\Sigma w-\mu$);
4. impose les contraintes : chaque poids entre 0 et 1 (*long-only*) et somme des poids égale à 1 (portefeuille entièrement investi) ;
 5. résout avec SLSQP en partant du portefeuille équipondéré.
 <!-- La tolérance `ftol=1e-12` est indispensable : avec la tolérance par défaut et des variances journalières de l'ordre de $10^{-4}$, le solveur s'arrête dès le point de départ et renvoie 1/N ; -->
6. remet à zéro les poids très légèrement négatifs, puis renormalise pour que la somme fasse exactement 1.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

def optimize_markowitz(returns_window: pd.DataFrame, gamma: float = 5.0) -> np.ndarray:
    """
    Calcule l'allocation optimale de Markowitz (Moyenne-Variance) sous contrainte long-only.

    L'optimisation maximise l'utilité de l'investisseur : U(w) = w^T * mu - (gamma / 2) * w^T * Sigma * w.

    Paramètres
    ----------
    returns_window : pd.DataFrame
        Matrice des rendements journaliers sur la fenêtre d'investissement (T jours x N actifs).
    gamma : float, optionnel
        Coefficient d'aversion au risque (défaut: 5.0).

    Retourne
    -------
    np.ndarray
        Vecteur des pondérations optimales (somme = 1, valeurs >= 0).
    """
    n_assets = returns_window.shape[1]

    # 1. Estimation des moments statistiques (annualisés sur une base de 252 jours)
    mu = returns_window.mean().values * 252
    cov_matrix = returns_window.cov().values * 252

    # 2. Définition de la fonction objectif (minimisation de l'opposé de l'utilité)
    def objective(w: np.ndarray) -> float:
        return (gamma / 2) * w.T @ cov_matrix @ w - mu.T @ w

    # Définition de la Jacobienne pour accélérer la convergence du solveur
    def jacobian(w: np.ndarray) -> np.ndarray:
        return gamma * cov_matrix @ w - mu

    # 3. Contraintes et bornes (Long-only et portefeuille totalement investi)
    bounds = tuple((0.0, 1.0) for _ in range(n_assets))
    constraints = {
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1.0,
        "jac": lambda w: np.ones(n_assets)
    }

    # 4. Résolution de l'optimisation
    # Note : La tolérance (ftol) est abaissée à 1e-12 pour forcer la convergence
    # face à des variances journalières très faibles.
    initial_weights = np.full(n_assets, 1.0 / n_assets)

    result = minimize(
        fun=objective,
        x0=initial_weights,
        jac=jacobian,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )

    # 5. Nettoyage des pondérations (élimination du bruit numérique résiduel du solveur)
    optimal_weights = np.clip(result.x, 0.0, None)

    return optimal_weights / np.sum(optimal_weights)




**`generate_historical_targets(returns, horizon, gamma)`** applique la fonction précédente à chaque date $t$ de l'historique, sur la fenêtre **future** `returns.iloc[t+1 : t+horizon+1]`, c'est-à-dire les jours $t+1$ à $t+H$.

**Remarques**
-  Les $H$ dernières dates n'ont pas de fenêtre future complète : elles restent à `NaN`.

- Le k-NN utilise que pour des dates passées $s$ dont la fenêtre $[s+1, s+H]$ est entièrement terminée à la date de décision $t$ (condition $s+H<t$).



In [ ]:
def generate_historical_targets(returns: pd.DataFrame, horizon: int, gamma: float = 5.0) -> pd.DataFrame:
    """
    Calcule les allocations optimales cibles pour chaque date historique.

    Pour chaque date t, l'algorithme observe la fenêtre future [t+1, t+horizon]
    pour déterminer l'allocation qui aurait été optimale a posteriori.

    Paramètres
    ----------
    returns : pd.DataFrame
        Historique complet des rendements journaliers des actifs.
    horizon : int
        Taille de la fenêtre d'évaluation future (H).
    gamma : float, optionnel
        Coefficient d'aversion au risque.

    Retourne
    -------
    pd.DataFrame
        Matrice des allocations cibles de même dimension que `returns`.
        Les `horizon` dernières lignes contiennent des NaN (fenêtre future incomplète).
    """
    n_days, n_assets = returns.shape
    targets = np.full((n_days, n_assets), np.nan)

    # Itération sur l'historique (en excluant la fin où la fenêtre H n'est pas complète)
    for t in range(n_days - horizon):
        future_window = returns.iloc[t + 1 : t + horizon + 1]
        targets[t] = optimize_markowitz(future_window, gamma=gamma)

    targets_df = pd.DataFrame(targets, index=returns.index, columns=returns.columns)

    return targets_df



## 5. L'algorithme k-NN :

### 5.1 Ce que fait l'algorithme à une date de décision $t$

À la clôture de $t$, un gérant doit choisir l'allocation qu'il détiendra à partir de $t+1$. L'algorithme procède en six étapes.

**Étape 1 — Dates candidates.** Seules les dates passées $s$ dont la cible est entièrement connue peuvent servir de voisins : $s + H < t$.
<!--  En positions (lignes de `M["X"]`), ce sont les lignes $0$ à $t-H-1$. Si moins de ~250 candidats sont disponibles (début de l'historique), la date est ignorée. -->

**Étape 2 — Mise à l'échelle.** On prend toutes les lignes de $X$ jusqu'à $t$ inclus. Pour chaque colonne $j$, on calcule la moyenne $m_j$ et l'écart-type $s_j$ **sur ces lignes seulement**, et on standardise :
$$z_{j,u}=\frac{x_{j,u}-m_j}{s_j},\qquad u\le t .$$
On écrête à $\pm 4$, pour qu'une journée extrême ne domine pas les distances, puis on multiplie chaque colonne par $1/\sqrt{n_{\text{famille}}}$ ($n$ = 10 pour Ret20, Vol60 et EMA10_50, 4 pour FF).

**Étape 3 — Distances.** On calcule la distance entre $z_t$ (la ligne de la date $t$) et chacune des lignes candidates, avec la métrique choisie :
- euclidienne : `cdist(..., "euclidean")` ;
- Manhattan : `cdist(..., "cityblock")` ;
- Mahalanobis : on estime $\Sigma$ sur les lignes standardisées jusqu'à $t$ avec `LedoitWolf().fit(...)`, on l'inverse, puis `cdist(..., "mahalanobis", VI=Σ⁻¹)`.

**Étape 4 — Sélection des $k$ voisins avec écart minimal.** On parcourt les candidats du plus proche au plus lointain, et on retient un candidat seulement s'il est à au moins `MIN_GAP` = 10 séances de tous les voisins déjà retenus. On s'arrête à $k$ voisins.
*Pourquoi* : deux jours consécutifs ont presque le même état (fenêtres glissantes). Sans cette règle, les $k$ voisins seraient $k$ jours d'un même épisode, soit en réalité un seul exemple.

**Étape 5 — Poids des voisins.** $\alpha_j = \dfrac{1/(d_j+\varepsilon)}{\sum_l 1/(d_l+\varepsilon)}$, avec $\varepsilon$ très petit pour éviter une division par zéro. Plus un voisin est proche, plus il pèse.

**Étape 6 — Allocation.** $\hat w_t=\sum_j \alpha_j\, w^*_{s_j}$. C'est automatiquement un portefeuille valide : poids $\ge 0$ et somme égale à 1, car c'est une moyenne pondérée de portefeuilles valides.

L'allocation $\hat w_t$ est conservée jusqu'à la date de rééquilibrage suivante, 5 séances plus tard (`REB`).


### 5.3 Pièges à éviter

1. **Information future dans la standardisation** : ne jamais calculer la moyenne et l'écart-type sur tout l'échantillon. Seulement sur les lignes jusqu'à $t$.
2. **Information future dans les voisins** : vérifier la condition $s+H<t$. Un voisin dont la fenêtre future n'est pas terminée « connaît » l'avenir de la date de décision.
3. **Décalage d'un jour** : l'allocation décidée à la clôture de $t$ ne s'applique qu'aux rendements de $t+1$ (`shift(1)`).
4. **Temps de calcul** : ne jamais rappeler `optimize_markowitz` dans la boucle du k-NN ; les cibles sont déjà calculées au §4.1.
5. **Comparabilité** : toutes les stratégies doivent être évaluées sur exactement les mêmes jours.

In [ ]:
# ===================== k-NN, baselines et évaluation =====================
def select_neighbors(dist, k, gap):
    "Les k plus proches, en imposant un écart >= gap séances entre voisins retenus."
    chosen = []
    for s in np.argsort(dist):
        if all(abs(s - c) >= gap for c in chosen):
            chosen.append(s)
            if len(chosen) == k:
                break
    return np.array(chosen)


def knn_weights(X, W_star, dates, k, metric, H, gap=MIN_GAP, weighting="inv", random=False, seed=SEED):
    "Allocation k-NN à chaque date de rééquilibrage (une ligne par date)."
    rng = np.random.default_rng(seed)
    Xv, Wv = X.values, W_star.values
    fam = X.columns.str.split("|").str[0]
    col_w = 1 / np.sqrt(fam.map(fam.value_counts()).values.astype(float))   # chaque famille pèse autant
    out = {}
    for t in X.index.get_indexer(dates):
        n_cand = t - H                                   # candidats s = 0..t-H-1  (s + H < t)
        if random:                                       # témoin : voisins tirés au hasard, poids égaux
            idx = rng.choice(n_cand, k, replace=False)
            a = np.ones(k)
        else:
            hist = Xv[:t + 1]                            # historique disponible à la clôture de t
            Z = np.clip((hist - hist.mean(0)) / (hist.std(0) + 1e-12), -4, 4) * col_w
            if metric == "mahalanobis":
                VI = np.linalg.inv(LedoitWolf().fit(Z).covariance_)
                dist = cdist(Z[t:t + 1], Z[:n_cand], "mahalanobis", VI=VI)[0]
            else:
                dist = cdist(Z[t:t + 1], Z[:n_cand], {"euclidienne": "euclidean", "manhattan": "cityblock"}[metric])[0]
            idx = select_neighbors(dist, k, gap)
            a = 1 / (dist[idx] + 1e-8) if weighting == "inv" else np.ones(len(idx))
        out[X.index[t]] = (a / a.sum()) @ Wv[idx]        # moyenne pondérée des allocations optimales
    return pd.DataFrame(out, index=W_star.columns).T


def reb_dates(name, period):
    "Dates de rééquilibrage de la période ; au moins MIN_CAND candidats pour le plus grand H (départ commun)."
    idx = MARKETS[name]["X"].index
    ok = (idx >= period[0]) & (idx <= period[1]) & (np.arange(len(idx)) >= MIN_CAND + max(H_GRID))
    return idx[ok][::REB]


def run_knn(name, period, k, metric, H, label="mv", drop=None, **kw):
    "k-NN sur une période ; 'drop' retire une famille de variables (ablation)."
    X = MARKETS[name]["X"]
    if drop is not None:
        X = X.loc[:, ~X.columns.str.startswith(drop)]
    return knn_weights(X, get_labels(name, label, H), reb_dates(name, period), k, metric, H, **kw)


def run_baseline(name, period, kind):
    "CRP 1/N ou Markowitz glissant (variance minimale sur les 60 dernières séances, connues en t)."
    R = MARKETS[name]["R"]
    out = {}
    for t in R.index.get_indexer(reb_dates(name, period)):
        out[R.index[t]] = (np.full(R.shape[1], 1 / R.shape[1]) if kind == "crp"
                           else markowitz(R.values[t - 59:t + 1], "minvar"))
    return pd.DataFrame(out, index=R.columns).T


def evaluate(W_reb, R, end, fee=0.0):
    "Performance d'allocations décidées à la clôture de t, appliquées à partir de t+1 ; fee = frais proportionnels."
    W = W_reb.reindex(R.index).ffill().shift(1)
    turn = W_reb.diff().abs().sum(axis=1)
    turn.iloc[0] = 1.0                                   # achat initial du portefeuille
    cost = (fee * turn).reindex(R.index).shift(1).fillna(0.0)   # frais payés au moment du rééquilibrage
    pr = ((W * R).sum(axis=1, min_count=1) - cost).loc[:end].dropna()
    wealth = (1 + pr).cumprod()
    stats = {"Rend. ann.": wealth.iloc[-1] ** (252 / len(pr)) - 1,
             "Vol. ann.": pr.std() * np.sqrt(252),
             "Sharpe": pr.mean() / pr.std() * np.sqrt(252),
             "Max DD": (wealth / wealth.cummax() - 1).min(),
             "Rotation": turn.iloc[1:].mean() / 2}
    return stats, wealth

## 6. Choix des paramètres sur la période de validation (Output 3)

Grille testée : $k \in \{5, 10, 20, 40\}$ × distance ∈ {euclidienne, Manhattan, Mahalanobis} × $H \in \{20, 60\}$, soit 24 combinaisons par marché.
Critère : **ratio de Sharpe sur la période de validation**. Les baselines sont calculées sur la même période pour comparaison.

In [ ]:
# ===================== Recherche de grille (validation) =====================
best, grids, rows_best = {}, {}, []
for name, M in MARKETS.items():
    rows = []
    for H in H_GRID:
        for metric in METRICS:
            for k in K_GRID:
                st, _ = evaluate(run_knn(name, M["val"], k, metric, H), M["R"], M["val"][1])
                rows.append(dict(H=H, distance=metric, k=k, **st))
    g = pd.DataFrame(rows)
    grids[name] = g
    g.to_csv(f"{RES_DIR}/grille_validation_{name}.csv", index=False)
    b = g.loc[g["Sharpe"].idxmax()]
    best[name] = (int(b["k"]), b["distance"], int(b["H"]))
    crp, _ = evaluate(run_baseline(name, M["val"], "crp"), M["R"], M["val"][1])
    mkw, _ = evaluate(run_baseline(name, M["val"], "minvar"), M["R"], M["val"][1])
    rows_best.append({"Marché": name, "k optimal": best[name][0], "Distance": best[name][1],
                      "H optimal": best[name][2], "Sharpe k-NN (val.)": b["Sharpe"],
                      "Sharpe CRP (val.)": crp["Sharpe"], "Sharpe Markowitz glissant (val.)": mkw["Sharpe"]})
    print(f"\n===== {name} : Sharpe en validation (lignes : H, distance ; colonnes : k) =====")
    display(g.pivot_table(index=["H", "distance"], columns="k", values="Sharpe"))
    print("Sharpe moyen par distance (toutes valeurs de k et H) :",
          g.groupby("distance")["Sharpe"].mean().round(3).to_dict())
    print("Sharpe moyen par k (toutes distances et H)            :",
          g.groupby("k")["Sharpe"].mean().round(3).to_dict())

params = pd.DataFrame(rows_best).set_index("Marché")
params.to_csv(f"{RES_DIR}/parametres_optimaux.csv")
print("\n===== Tableau des paramètres optimaux =====")
display(params)

TypeError: generate_historical_targets() takes from 2 to 3 positional arguments but 4 were given

## 7. Résultats hors échantillon

Les paramètres retenus en validation sont appliqués **sans modification** à la période de test (2018-2025). On compare, sur exactement les mêmes jours :
- le k-NN avec les paramètres optimaux ;
- le CRP 1/N ;
- le Markowitz glissant 60 jours ;
- le k-NN aléatoire (témoin).

**Livrables attendus**
1. Un tableau des indicateurs (rendement, volatilité, Sharpe, drawdown maximal, rotation) par marché et par stratégie.
2. **Les courbes de croissance du capital** ($V_t$, base 1), un graphique par marché : c'est l'Output 1 demandé par le sujet.
3. Un graphique empilé des allocations du k-NN dans le temps (`stackplot`), qui montre comment le portefeuille se déplace d'un secteur à l'autre selon le régime. Il est très utile en soutenance.

**Lecture** : regarder où l'écart entre le k-NN et le CRP se crée (Covid 2020, inflation et hausse des taux 2022, régulation de la tech chinoise 2021, relance chinoise 2024), et si le k-NN réduit le drawdown pendant ces crises.

In [ ]:
# ===================== Période de test : k-NN vs baselines =====================
perf_test, wealth_test, alloc_test = [], {}, {}
for name, M in MARKETS.items():
    k, metric, H = best[name]
    strategies = {f"k-NN (k={k}, {metric}, H={H})": run_knn(name, M["test"], k, metric, H),
                  "CRP 1/N": run_baseline(name, M["test"], "crp"),
                  "Markowitz glissant 60 j": run_baseline(name, M["test"], "minvar"),
                  "k-NN aléatoire (témoin)": run_knn(name, M["test"], k, metric, H, random=True)}
    wealth_test[name] = {}
    for strat, W in strategies.items():
        st, wealth_test[name][strat] = evaluate(W, M["R"], M["test"][1])
        perf_test.append({"Marché": name, "Stratégie": strat, **st})
    alloc_test[name] = strategies[f"k-NN (k={k}, {metric}, H={H})"]

perf_test = pd.DataFrame(perf_test).set_index(["Marché", "Stratégie"])
perf_test.to_csv(f"{RES_DIR}/performances_test.csv")
display(perf_test)

# --- Courbes de croissance du capital ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, name in zip(axes, MARKETS):
    for strat, w in wealth_test[name].items():
        ax.plot(w, label=strat, lw=2 if strat.startswith("k-NN (") else 1.2,
                ls="--" if "CRP" in strat else "-")
    ax.set_title(f"{name} — croissance du capital (période de test)")
    ax.set_ylabel("Capital (base 1)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{RES_DIR}/courbes_capital_test.png", dpi=150)
plt.show()

# --- Allocations du k-NN dans le temps ---
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, name in zip(axes, MARKETS):
    W = alloc_test[name]
    ax.stackplot(W.index, W.T.values, labels=W.columns)
    ax.set_title(f"{name} — allocations k-NN")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc="upper left", ncol=2)
plt.tight_layout()
plt.savefig(f"{RES_DIR}/allocations_knn_test.png", dpi=150)
plt.show()

#Commentaire

## 8. Étude d'ablation (Output 2)

On repart des paramètres retenus en validation et on modifie **un seul élément à la fois**, sur la période de test :
- retrait de chacune des 4 familles de variables du vecteur d'état (rendements 20 j, volatilités 60 j, ratios EMA, facteurs de style) ;
- choix de conception de l'algorithme : sans écart minimal entre voisins, poids uniformes au lieu de l'inverse de la distance, cible de variance minimale au lieu de moyenne-variance ;
- témoin : voisins tirés au hasard (moyenne sur 5 tirages).

**Lecture** : si retirer une famille ne dégrade pas le Sharpe (ou l'améliore), cette famille n'est pas nécessaire et la solution minimale devrait l'écarter. C'est précisément l'objet de l'étude d'ablation demandée par le sujet.

In [ ]:
# ===================== Ablation =====================
FAMILIES = {"Ret20": "rendements 20 j", "Vol60": "volatilités 60 j",
            "EMA10_50": "ratios EMA10/EMA50", "FF": "facteurs de style"}


def ablation(name):
    M = MARKETS[name]
    k, metric, H = best[name]
    variants = {"Complet": {}}
    variants.update({f"Sans {lab}": {"drop": f} for f, lab in FAMILIES.items()
                     if M["X"].columns.str.startswith(f).any()})
    variants.update({"Sans écart entre voisins": {"gap": 1},
                     "Poids uniformes": {"weighting": "uniform"},
                     "Cible variance min.": {"label": "minvar"}})
    res = {v: evaluate(run_knn(name, M["test"], k, metric, H, **kw), M["R"], M["test"][1])[0]
           for v, kw in variants.items()}
    rnd = [evaluate(run_knn(name, M["test"], k, metric, H, random=True, seed=SEED + i),
                    M["R"], M["test"][1])[0] for i in range(5)]
    res["Voisins aléatoires (moy. 5 tirages)"] = pd.DataFrame(rnd).mean().to_dict()
    return pd.DataFrame(res).T


abl = {name: ablation(name) for name in MARKETS}
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, name in zip(axes, MARKETS):
    a = abl[name]
    a.to_csv(f"{RES_DIR}/ablation_{name}.csv")
    print(f"\n===== Ablation — {name} (période de test) =====")
    display(a)
    colors = ["tab:blue"] + ["tab:orange"] * (len(a) - 2) + ["tab:gray"]
    ax.barh(a.index[::-1], a["Sharpe"][::-1], color=colors[::-1])
    ax.axvline(perf_test.loc[(name, "CRP 1/N"), "Sharpe"], color="k", ls="--", label="CRP 1/N")
    ax.set_title(f"{name} — Sharpe selon la variante")
    ax.legend()
plt.tight_layout()
plt.savefig(f"{RES_DIR}/ablation_test.png", dpi=150)
plt.show()

## 9. Analyse critique et conclusion *(à rédiger à partir des résultats)*

**Questions auxquelles la conclusion doit répondre**
1. Sur chaque marché, le k-NN bat-il le CRP **et** le Markowitz glissant en Sharpe et en drawdown ?
2. Le k-NN bat-il nettement le **témoin aléatoire** ? Sinon, la similarité des états n'apporte pas d'information, et le gain éventuel vient seulement de la moyenne d'allocations optimales.
3. Quelles familles de variables sont nécessaires d'après l'ablation ? Une famille inutile doit être retirée de la solution finale.
4. La rotation est-elle raisonnable ? Avec des frais de 0,1 % par transaction, que deviendrait le Sharpe ?
5. Les deux marchés donnent-ils les mêmes conclusions ? Les univers ont la même structure (10 actions, 10 secteurs) : un écart renvoie à la nature du marché (émergent, plus volatil, plus sensible à la politique économique), pas au type d'actif.

**Limites à mentionner**
- biais du survivant (actions choisies aujourd'hui) ;
- pas de frais de transaction ;
- facteurs de style approximés à partir de 10 titres, avec un classement *value*/*growth* fixe ;
- cibles estimées avec une covariance empirique sur peu de jours (Ledoit-Wolf serait plus robuste) ;
- un seul découpage validation/test (une validation glissante serait plus solide).

# Commentaire

In [ ]:
# ===================== Sauvegarde des données et résultats =====================
with zipfile.ZipFile("donnees_projet3.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (DATA_DIR, RES_DIR):
        for f in sorted(os.listdir(folder)):
            z.write(os.path.join(folder, f))
print("Archive créée : donnees_projet3.zip (data/ + results/)")